In [9]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report, confusion_matrix

In [10]:
# Load and combine
df_2023 = pd.read_csv('../data/StormEvents_details-ftp_v1.0_d2023_c20260323.csv')
df_2024 = pd.read_csv('../data/StormEvents_details-ftp_v1.0_d2024_c20260323 2.csv')
df = pd.concat([df_2023, df_2024], ignore_index=True)

# Filter to top 10 event types
top_10_types = df['EVENT_TYPE'].value_counts().head(10).index.tolist()
print(df['EVENT_TYPE'].value_counts().head(10))
df = df[df['EVENT_TYPE'].isin(top_10_types)].copy()
# Feature engineering (all vectorized, no loops)
month_map = {'January':1, 'February':2, 'March':3, 'April':4, 'May':5, 'June':6,
             'July':7, 'August':8, 'September':9, 'October':10, 'November':11, 'December':12}
df['MONTH'] = df['MONTH_NAME'].map(month_map)

# Hour from BEGIN_TIME (stored as HHMM integer, e.g. 1430 -> 14)
df['HOUR'] = df['BEGIN_TIME'] // 100

# Duration in minutes from full timestamps
df['BEGIN_DT'] = pd.to_datetime(df['BEGIN_DATE_TIME'], format='%d-%b-%y %H:%M:%S')
df['END_DT'] = pd.to_datetime(df['END_DATE_TIME'], format='%d-%b-%y %H:%M:%S')
df['DURATION_MIN'] = (df['END_DT'] - df['BEGIN_DT']).dt.total_seconds() / 60

# CZ_TYPE to binary
df['CZ_TYPE_ENC'] = (df['CZ_TYPE'] == 'C').astype(int)

# Label encode state 
le_state = LabelEncoder()
df['STATE_ENC'] = le_state.fit_transform(df['STATE'].astype(str))

# Label encode WFO
le_wfo = LabelEncoder()
df['WFO_ENC'] = le_wfo.fit_transform(df['WFO'].astype(str))

# Select features and target
feature_cols = ['MONTH', 'HOUR', 'STATE_ENC', 'BEGIN_LAT', 'BEGIN_LON',
                'DURATION_MIN', 'CZ_TYPE_ENC', 'WFO_ENC']
X = df[feature_cols]
y = df['EVENT_TYPE']

# Sanity checks
print(f"\nShape: {X.shape}")
print(f"Duration stats:\n{df['DURATION_MIN'].describe()}")
print(f"Missing per column:\n{X.isna().sum()}")
print(df.groupby('EVENT_TYPE')['DURATION_MIN'].describe())

# Encode target labels (XGBoost wants integer classes)
le_y = LabelEncoder()
y_enc = le_y.fit_transform(y)

# Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, stratify=y_enc, test_size=0.2, random_state=42
)

EVENT_TYPE
Thunderstorm Wind           41177
Hail                        20702
Drought                      9288
Flash Flood                  8286
Excessive Heat               7684
High Wind                    7485
Heat                         6494
Winter Weather               6448
Flood                        5318
Marine Thunderstorm Wind     4835
Name: count, dtype: int64

Shape: (117717, 8)
Duration stats:
count    117717.000000
mean       3332.777738
std       10449.713745
min           0.000000
25%           0.000000
50%           2.000000
75%         420.000000
max       44639.000000
Name: DURATION_MIN, dtype: float64
Missing per column:
MONTH               0
HOUR                0
STATE_ENC           0
BEGIN_LAT       37399
BEGIN_LON       37399
DURATION_MIN        0
CZ_TYPE_ENC         0
WFO_ENC             0
dtype: int64
                            count          mean           std  min      25%  \
EVENT_TYPE                                                                    
D

In [11]:
# Train default XGBoost
model = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=10,
    random_state=42,
    tree_method='hist'  # fast, handles NaN natively
)
model.fit(X_train, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softmax'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method

In [12]:
# Evaluate
y_pred = model.predict(X_test)
macro_f1 = f1_score(y_test, y_pred, average='macro')
print(f"\nMacro F1: {macro_f1:.4f}")
print(classification_report(y_test, y_pred, target_names=le_y.classes_))


Macro F1: 0.8891
                          precision    recall  f1-score   support

                 Drought       1.00      1.00      1.00      1858
          Excessive Heat       0.93      0.82      0.87      1537
             Flash Flood       0.87      0.86      0.87      1657
                   Flood       0.87      0.80      0.83      1064
                    Hail       0.74      0.65      0.69      4140
                    Heat       0.81      0.93      0.86      1299
               High Wind       0.97      0.95      0.96      1497
Marine Thunderstorm Wind       1.00      1.00      1.00       967
       Thunderstorm Wind       0.82      0.88      0.85      8235
          Winter Weather       0.94      0.97      0.95      1290

                accuracy                           0.86     23544
               macro avg       0.90      0.89      0.89     23544
            weighted avg       0.86      0.86      0.86     23544



In [13]:
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=le_y.classes_, columns=le_y.classes_)
print(cm_df)
imp = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(imp)

                          Drought  Excessive Heat  Flash Flood  Flood  Hail  \
Drought                      1854               3            0      0     0   
Excessive Heat                  3            1253            0      0     0   
Flash Flood                     0               0         1429    104    24   
Flood                           0               0          167    850    10   
Hail                            0               0           12      6  2705   
Heat                            2              95            0      0     0   
High Wind                       0               1            0      0     0   
Marine Thunderstorm Wind        0               0            0      0     0   
Thunderstorm Wind               0               0           26     13   920   
Winter Weather                  1               0            0      0     0   

                          Heat  High Wind  Marine Thunderstorm Wind  \
Drought                      1          0                  

## Hierarchical Lat/Lon Imputation
Fill missing `BEGIN_LAT`/`BEGIN_LON` using progressively coarser geographic groups:
1. **(STATE, CZ_FIPS)** — county/zone median (~20 mi precision)
2. **(STATE, WFO)** — Weather Forecast Office region median (~100 mi)
3. **STATE** — state-level median (last resort)

In [ ]:
# Hierarchical imputation: compute median lat/lon at each level from TRAINING data only
# We need the original df (before split) to have CZ_FIPS and WFO columns for grouping,
# so we rebuild X with imputed coords.

# Step 1: Compute group medians from training rows that have coords
train_idx = X_train.index
df_train = df.loc[train_idx]
has_coords_train = df_train[df_train['BEGIN_LAT'].notna()]

median_cz = has_coords_train.groupby(['STATE', 'CZ_FIPS'])[['BEGIN_LAT', 'BEGIN_LON']].median()
median_wfo = has_coords_train.groupby(['STATE', 'WFO'])[['BEGIN_LAT', 'BEGIN_LON']].median()
median_state = has_coords_train.groupby('STATE')[['BEGIN_LAT', 'BEGIN_LON']].median()

# Step 2: Apply hierarchical fill to the full dataframe
df_imputed = df.copy()

# Track which level filled each row
df_imputed['IMPUTE_LEVEL'] = 'original'
missing_mask = df_imputed['BEGIN_LAT'].isna()
df_imputed.loc[missing_mask, 'IMPUTE_LEVEL'] = 'unfilled'

# Level 1: (STATE, CZ_FIPS)
for (state, cz_fips), row in median_cz.iterrows():
    mask = missing_mask & (df_imputed['STATE'] == state) & (df_imputed['CZ_FIPS'] == cz_fips)
    df_imputed.loc[mask, 'BEGIN_LAT'] = row['BEGIN_LAT']
    df_imputed.loc[mask, 'BEGIN_LON'] = row['BEGIN_LON']
    df_imputed.loc[mask, 'IMPUTE_LEVEL'] = 'L1_county_zone'

# Update missing mask
missing_mask = df_imputed['BEGIN_LAT'].isna()

# Level 2: (STATE, WFO)
for (state, wfo), row in median_wfo.iterrows():
    mask = missing_mask & (df_imputed['STATE'] == state) & (df_imputed['WFO'] == wfo)
    df_imputed.loc[mask, 'BEGIN_LAT'] = row['BEGIN_LAT']
    df_imputed.loc[mask, 'BEGIN_LON'] = row['BEGIN_LON']
    df_imputed.loc[mask, 'IMPUTE_LEVEL'] = 'L2_wfo'

# Update missing mask
missing_mask = df_imputed['BEGIN_LAT'].isna()

# Level 3: STATE
for state, row in median_state.iterrows():
    mask = missing_mask & (df_imputed['STATE'] == state)
    df_imputed.loc[mask, 'BEGIN_LAT'] = row['BEGIN_LAT']
    df_imputed.loc[mask, 'BEGIN_LON'] = row['BEGIN_LON']
    df_imputed.loc[mask, 'IMPUTE_LEVEL'] = 'L3_state'

# Report
print("Imputation coverage:")
print(df_imputed['IMPUTE_LEVEL'].value_counts())
print(f"\nRemaining NaN lat: {df_imputed['BEGIN_LAT'].isna().sum()}")
print(f"Remaining NaN lon: {df_imputed['BEGIN_LON'].isna().sum()}")

In [ ]:
# Rebuild X from imputed df using the same feature columns and split indices
X_imputed = df_imputed[feature_cols]
X_train_imp = X_imputed.loc[train_idx]
X_test_imp = X_imputed.loc[X_test.index]

print(f"Imputed train NaNs: {X_train_imp.isna().sum().sum()}")
print(f"Imputed test NaNs:  {X_test_imp.isna().sum().sum()}")

# Train XGBoost on imputed data (same hyperparams as baseline)
model_imp = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=10,
    random_state=42,
    tree_method='hist'
)
model_imp.fit(X_train_imp, y_train)

# Evaluate
y_pred_imp = model_imp.predict(X_test_imp)
macro_f1_imp = f1_score(y_test, y_pred_imp, average='macro')
print(f"\nMacro F1 (imputed): {macro_f1_imp:.4f}")
print(classification_report(y_test, y_pred_imp, target_names=le_y.classes_))

In [ ]:
# Side-by-side comparison: NaN baseline vs imputed
from sklearn.metrics import f1_score as f1

classes = le_y.classes_
f1_nan = f1_score(y_test, y_pred, average=None)
f1_imp = f1_score(y_test, y_pred_imp, average=None)

comparison = pd.DataFrame({
    'Class': classes,
    'F1 (NaN)': f1_nan,
    'F1 (Imputed)': f1_imp,
    'Diff': f1_imp - f1_nan
}).set_index('Class')

print("=" * 60)
print("Per-class F1 comparison: NaN baseline vs Hierarchical Imputation")
print("=" * 60)
print(comparison.to_string(float_format='%.4f'))
print("-" * 60)
print(f"{'Macro F1 (NaN):':<30} {macro_f1:.4f}")
print(f"{'Macro F1 (Imputed):':<30} {macro_f1_imp:.4f}")
print(f"{'Difference:':<30} {macro_f1_imp - macro_f1:+.4f}")

# Feature importance comparison
imp_imputed = pd.Series(model_imp.feature_importances_, index=feature_cols).sort_values(ascending=False)
imp_comparison = pd.DataFrame({
    'Importance (NaN)': imp,
    'Importance (Imputed)': imp_imputed,
}).sort_values('Importance (NaN)', ascending=False)
print("\n" + "=" * 60)
print("Feature importance comparison")
print("=" * 60)
print(imp_comparison.to_string(float_format='%.4f'))